# 00 — Sentinel-2 export from Google Earth Engine

Cloud/snow-masked Sentinel-2 (Surface Reflectance, harmonized) monthly composites for June–October, with NDVI and NARI, exported per year and per mosaic to Google Drive. This produces the spectral inputs to the classification feature stack.

**Before running:** set your Earth Engine project ID in place of `YOUR-GEE-PROJECT-ID`. The study area is covered by two mosaics; run once per mosaic per year by toggling the `geometry` line in the parameter block. NDVI-phenology features and GLCM textures are produced in later steps (see README), not here.

_Manuscript: Methods — Sentinel-2 acquisition and preprocessing._

In [ ]:
import ee
import geemap

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='YOUR-GEE-PROJECT-ID')

# === PARAMETERS ===
year = 2025
months = [6, 7, 8, 9, 10]
#geometry = ee.Geometry.Rectangle([10.7, 47.39, 11.7, 47.67]) # small test area
#geometry = ee.Geometry.Rectangle([9.6, 47.25, 11.71, 48.0])  # mosaic 1
geometry = ee.Geometry.Rectangle([11.7, 47.25, 13.1, 48.0]) # mosaic 2
drive_folder = "GEE_Sentinel2_Exports"

# === LOAD COLLECTION ===
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(geometry) \
    .filter(ee.Filter.calendarRange(year, year, "year")) \
    .filter(ee.Filter.calendarRange(months[0], months[-1], "month")) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

# === CLOUD/SNOW MASKING ===
def mask_clouds_and_snow(image):
    cloud_mask = image.select('QA60').bitwiseAnd(1 << 10).eq(0)
    scl = image.select('SCL')
    snow_mask = scl.neq(11)  # snow
    cloud_scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask.And(snow_mask).And(cloud_scl_mask))

s2_masked = s2.map(mask_clouds_and_snow)

# === COMPUTE NDVI AND NARI ===
def compute_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    b3 = image.select('B3').multiply(0.0001).add(0.05)  # Green
    b5 = image.select('B5').resample('bilinear').multiply(0.0001).add(0.05)  # Red-edge
    nari = b3.pow(-1).subtract(b5.pow(-1)).divide(b3.pow(-1).add(b5.pow(-1))).rename('NARI')
    return image.addBands(ndvi).addBands(nari)

s2_indexed = s2_masked.map(compute_indices)

# === MONTHLY COMPOSITES ===
def monthly_composite(month):
    start = f"{year}-{str(month).zfill(2)}-01"
    end = f"{year}-{str(month).zfill(2)}-30"
    return s2_indexed.filterDate(start, end) \
        .select(['B2', 'B3', 'B4', 'B8', 'NDVI', 'NARI']) \
        .median() \
        .clip(geometry) \
        .set('month', month)

monthly_imgs = [monthly_composite(m) for m in months]
monthly_ic = ee.ImageCollection(monthly_imgs)

# === GAP FILL ===
def gap_fill(img):
    return img.unmask(monthly_ic.median())

filled_imgs = [gap_fill(img) for img in monthly_imgs]

# === EXPORT FUNCTION ===
band_names = {
    'B2': 'Blue', 'B3': 'Green', 'B4': 'Red', 'B8': 'NIR',
    'NDVI': 'NDVI', 'NARI': 'NARI'
}

def export_band_stack(band):
    stacked = ee.Image.cat([img.select(band) for img in filled_imgs])
    task = ee.batch.Export.image.toDrive(
        image=stacked,
        description=f"Sentinel2_{band_names[band]}_Stack_{year}_Filled",
        folder=drive_folder,
        fileNamePrefix=f"Sentinel2_{band_names[band]}_Stack_{year}_Filled",
        region=geometry,
        scale=10,
        maxPixels=1e13
    )
    task.start()
    print(f"Started export for {band_names[band]}.")

# === EXPORT ALL STACKS ===
for band in ['B2', 'B3', 'B4', 'B8', 'NDVI', 'NARI']:
    export_band_stack(band)

print("✅ All exports initiated.")
